● Understand your data - what do the features mean?
(May have to do some info gathering)
The features are all metrics that were collected to determine the likelihood of the patient having COVID-19. 

● Document data context and data sampling in markdown
However, nowhere in that dataset did it determine whetehr the patients were healthy or not. So we will be using it to determine and prove that the healthy patients are the one that survive.

● Explore and interpret data structure, descriptive
statistics, data quality, and variable relationships

● Explore data visually with appropriate visualizations

● Discuss and implement strategies for Handling Missing
Values, Removing Duplicates, and Handling Outliers

● Perform data transformation as appropriate

● Create at least one new feature and document your
approach

● Include a discussion around data quality assessment,
including data profiling, data completeness, data
accuracy, data consistency, data integrity, and data
lineage and provenance

● IMPORTANT: provide rationale for choices


Problem (What are we using the data for?): 
We are using the data to identify which patients are healthy or not.

In [2]:
import pandas as pd


In [ ]:
df_bp = pd.read_csv(r"C:\Users\Micha\Documents\Duke_Classes\AIPI_510\aipi510-fall25\data\week4\hrv-covid19\data\blood_pressure.csv")

df_hrv = pd.read_csv(r"C:\Users\Micha\Documents\Duke_Classes\AIPI_510\aipi510-fall25\data\week4\hrv-covid19\data\heart_rate.csv")
df_hrv
#df_bp


,user_code,datetime,heart_rate,is_resting
0,007b8190cf,2020-04-26 04:49:25,70,0
1,01bad5a519,2020-04-23 06:21:03,74,0
2,01bad5a519,2020-04-23 09:46:01,82,0
3,01bad5a519,2020-04-23 14:05:06,90,0
4,01bad5a519,2020-04-24 03:41:18,72,0
...,...,...,...,...
523778,fe5ca7e4ea,2020-05-23 06:31:33,85,0
523779,fe6c1b1349,2020-05-07 12:05:04,68,0
523780,fe6c1b1349,2020-05-10 06:32:00,70,0
523781,fe6c1b1349,2020-05-12 06:31:42,77,0


In [ ]:
exists = '007b8190cf' in df_bp['user_code'].astype(str).values #return False
exists = '007b8190cf' in df_bp['user_code'].astype(str).values #return True

print("Exists:", exists)

exists = '01bad5a519' in df_hrv['user_code'].astype(str).values #return False


Exists: False


I can tell from this that some users do not have all there data in both csv. According to https://www.heart.org/, some of the best measures to determine a persons health are heart rate 
and blood pressure. To keep things simple but still get the best picture of one's health
We are going to take the 

In [ ]:
#Now I need to combine both of these datasets and join based on usercode. There are 721 rows
#in blood pressure, and 523783 rows in heart rate. It stands to reasons that the user codes
#in blood pressure will probably be in heart rate.


In [40]:
# For blood pressure dataset
unique_bp_users = df_bp['user_code'].nunique()
print("Unique user_codes in df_bp:", unique_bp_users)

# For the other dataset (e.g., heart rate variability)
unique_hrv_users = df_hrv['user_code'].nunique()
print("Unique user_codes in df_hrv:", unique_hrv_users)

Unique user_codes in df_bp: 28
Unique user_codes in df_hrv: 79


In [ ]:
#df_bp_clean = df_bp.drop_duplicates(subset='user_code')
#print(df_bp_clean.shape)
#df_hrv_clean = df_hrv.drop_duplicates(subset='user_code')
#print(df_hrv_clean.shape)
#merged_df = pd.merge(df_bp_clean, df_hrv_clean, on='user_code', how='inner')
#print(merged_df.shape)


(28, 8)
(79, 4)
(28, 11)


In [47]:
# Drop rows where any of the key vitals are missing
#clean_df = merged_df.dropna(subset=['systolic', 'diastolic', 'heart_rate'])
#clean_df

#This doesn't work because we are dropping all the duplicates user codes although they are taken at different times.
# One option is  takign the average of them. However this still doesn't give us an accurate pciture of their health. 
# Knopwing that they have COVID, If we are takling the average heart rate or systolic blood pressure then, 
# because you could average the time that they were in great healthj and the time when their hea;lth sucked and 
# determine that they are ok. Any aggregation measure would not give an accurate picture. However if I took the last
#known heart rate and systolic bp, then that assumes worst case scenario, because we know they have COVID. So I made all
# duplicates take the last known measures to give the most updated measurement of their health.

# Ensure datetime format
df_bp['measurement_datetime'] = pd.to_datetime(df_bp['measurement_datetime'])

# Sort and keep the latest record per user
df_bp_latest = df_bp.sort_values('measurement_datetime').groupby('user_code', as_index=False).tail(1)
df_bp_latest.shape

# Ensure datetime format
df_hrv['datetime'] = pd.to_datetime(df_hrv['datetime'])

# Sort and keep the latest record per user
df_hrv_latest = df_hrv.sort_values('datetime').groupby('user_code', as_index=False).tail(1)

df_hrv_latest



,user_code,datetime,heart_rate,is_resting
438754,cdfbcad405,2020-02-03 19:13:13,68,0
7993,19021f3a0a,2020-03-26 06:02:56,95,0
298335,6ce06ce747,2020-03-28 11:43:45,69,0
460412,e1578ff865,2020-04-09 09:33:57,78,0
7984,128d2c7888,2020-04-10 04:42:22,60,0
...,...,...,...,...
461929,f9edcb7056,2020-06-19 02:22:26,79,0
511497,fcf3ea75b0,2020-06-19 03:45:38,64,0
328496,a1c2e6b2eb,2020-06-19 04:48:05,65,0
8562,295ed96279,2020-06-19 06:08:26,59,0


In [62]:
#Now that I have this, I can combine them based on user code:
merged_df = pd.merge(df_bp_latest, df_hrv_latest, on='user_code', how='inner')
print(merged_df)

     user_code measurement_datetime  diastolic  systolic  \
0   cdfbcad405  2019-12-31 10:48:51         90       151   
1   0d297d2410  2020-01-13 20:46:12         82       127   
2   295ed96279  2020-04-03 12:18:14         70       100   
3   7ba5381254  2020-04-17 13:59:00         96       153   
4   588d6b2046  2020-04-20 04:50:46         83       132   
5   9871ee5e7b  2020-04-21 10:42:39         82       157   
6   ad41d5b79c  2020-04-23 09:11:29         74       112   
7   7d2c87fb7e  2020-04-29 00:02:56         80       120   
8   f8b552df37  2020-05-02 16:24:39         80       120   
9   1b9321b648  2020-05-04 10:55:25         70       110   
10  982ec78569  2020-05-04 18:56:45         65       110   
11  425969dc69  2020-05-04 22:42:10         76       114   
12  8633d50fa7  2020-05-08 20:14:00         98       144   
13  5108b04245  2020-05-10 13:53:48         25        63   
14  01bad5a519  2020-05-12 02:09:32         80       120   
15  974d68bcdd  2020-05-12 13:50:00     

I know from my research that I am trying to create a new column which multiplies the HRV and the systolic blood pressure to give me the rate pressure product, a clinically validated formula that estimates myocardial oxygen demand. Basically it measures how hard the heart is working. So this next part is the feature engineering part where I combine the both of them into one feature to show their overall health.


In [63]:
merged_df['RPP'] = merged_df['systolic'] * merged_df['heart_rate']
merged_df.drop(['robinson_index', 'functional_changes_index', 'circulatory_efficiency', 'kerdo_vegetation_index', 
                'diastolic'], axis=1, inplace=True)
merged_df.head()

,user_code,measurement_datetime,systolic,datetime,heart_rate,is_resting,RPP
0,cdfbcad405,2019-12-31 10:48:51,151,2020-02-03 19:13:13,68,0,10268
1,0d297d2410,2020-01-13 20:46:12,127,2020-05-26 17:25:03,68,0,8636
2,295ed96279,2020-04-03 12:18:14,100,2020-06-19 06:08:26,59,0,5900
3,7ba5381254,2020-04-17 13:59:00,153,2020-05-06 01:50:25,88,0,13464
4,588d6b2046,2020-04-20 04:50:46,132,2020-04-20 11:50:29,74,0,9768


Now we know can see the person's general health based on RPP. From here, we want to determine whether they are actually healthy. 
We also know that they are all resting based off of the is_restign criteria when they took their heart rate. And we are making an assumption based on the timestamps that they were resting before they got their heart rate taken (even though there are two months in between lol so this would not be a valid assumption)

At this point, there is no time left but the only thing elft to do is identiofy whether they ae healthy or not based on what research says are healthy numbers.

Accroding to NCSF (from copilot):, A normal RPP is typically between 6,000 at rest and 40,000 during exercise. This range indicates a healthy cardiovascular function, as it reflects the efficient use of oxygen by the heart. An RPP above 10,000 suggests an increased risk for heart disease, while values below 16,000 may indicate insufficient cardiac function. So that would be the only thing left to do.

Documentation:

I used copilot to get me the sntax to read the files (I forgot to add the file.csv at the end of the file path lol). Then I used it to help get me sources for the most important measures. I also used its help to give me the syntax for figuring out the merge.
I used it again to give me the syntax for taking the last known systolic blood pressure and the last known heart rate for each of the unique user_codes.